Import necesarios

In [8]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prefect import task, flow
import sqlite3


Prueba de la API de frankfurter

In [9]:
url = "https://api.frankfurter.dev/v2/rates?from=2025-01-01&to=&to=2026-01-01&quotes=usd,gbp"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(url, headers=headers)

print(f"Código de estado devuelto: {response.status_code}")


if response.status_code == 200:
    datos = response.json()
    print("Tasas de cambio:")

else:
    print(f"Error en la conexión: {response.status_code}")

print("Extracción de datos completada.")

print(list(datos)[-10:-1])

Código de estado devuelto: 200
Tasas de cambio:
Extracción de datos completada.
[{'date': '2025-12-28', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.87236}, {'date': '2025-12-28', 'base': 'EUR', 'quote': 'USD', 'rate': 1.1784}, {'date': '2025-12-29', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.87254}, {'date': '2025-12-29', 'base': 'EUR', 'quote': 'USD', 'rate': 1.1774}, {'date': '2025-12-30', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.87136}, {'date': '2025-12-30', 'base': 'EUR', 'quote': 'USD', 'rate': 1.177}, {'date': '2025-12-31', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.87177}, {'date': '2025-12-31', 'base': 'EUR', 'quote': 'USD', 'rate': 1.1752}, {'date': '2026-01-01', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.87197}]


Función de extraccion de los datos

In [10]:

@task(retries=3, retry_delay_seconds=10)
def extract_data(url="https://api.frankfurter.dev/v2/rates", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, range_time=["2025-01-01", "2025-12-31"], currencies="usd,gbp", base="eur"):
    """
    Función para extraer datos de la API de Frankfurter con manejo de errores y reintentos.
    
    Parámetros:
    - url: URL de la API.
    - headers: Encabezados HTTP para la solicitud.
    - retries: Número de reintentos en caso de error.
    - delay: Tiempo de espera entre reintentos (en segundos).
    - range_time: Lista con las fechas de inicio y fin para la extracción de datos.
    
    Retorna:
    - Un DataFrame con los datos extraídos o None si falla la extracción.
    """

    complete_url = f"{url}?base={base}&quotes={currencies}&from={range_time[0]}&to={range_time[1]}"
    response = requests.get(complete_url, headers=headers)

    print(f"Código de estado devuelto: {response.status_code}")

    if response.status_code == 200:
        print("Datos extraidos con exito")
        datos = response.json()
        return datos

    else:
        print(f"Error en la conexión {response.status_code}")



Prueba

Limpieza de los datos

In [11]:
@task
def clean_data(data):

    data_df = pd.DataFrame(data)
    data_df['date'] = pd.to_datetime(data_df['date'])

    print("Datos limpiados con exito")
    return data_df


Funcion de carga en la bbdd

In [ ]:

@task
def load_data(df, db_name="historico_divisas.db", table_name="tasas_cambio"):
    """
    Carga el DataFrame limpio en una base de datos SQLite.
    """
    try:
        conn = sqlite3.connect(db_name)

        fecha_min = str(df['date'].min())
        fecha_max = str(df['date'].max())
        
        monedas_nuevas = df['quote'].unique().tolist()
        
        interrogantes = ", ".join(["?"] * len(monedas_nuevas))
        query_borrado = f"DELETE FROM {table_name} WHERE date >= ? AND date <= ? AND quote IN ({interrogantes})"
        
        parametros = [fecha_min, fecha_max] + monedas_nuevas
        
        cursor = conn.cursor()
        cursor.execute(query_borrado, parametros)
        conn.commit()

        df.to_sql(table_name, conn, if_exists='append', index=False)
        
        print(f"Carga exitosa: {len(df)} filas insertadas en la tabla '{table_name}'.")
        
    except Exception as e:
        print(f"Error al cargar los datos en la base de datos: {e}")
        
    finally:

        conn.close()

Orquestamos todo el flujo de trabajo con prefect

In [13]:
@flow(name="PipeLine-Divisas") 
def main_etl(url="https://api.frankfurter.dev/v2/rates", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, range_time=["2025-01-01", "2025-12-31"], currencies="usd,gbp", base="eur"):

    datos = extract_data(url=url, headers=headers, range_time=range_time, currencies=currencies, base=base)
    df = clean_data(datos)
    load_data(df)

In [14]:
main_etl(currencies="usd,gbp,jpy,cny,aud,cad", range_time=["2010-01-01", "2025-12-31"])

13:25:30.854 | INFO    | Flow run 'weightless-cuttlefish' - Beginning flow run 'weightless-cuttlefish' for flow 'PipeLine-Divisas'

Código de estado devuelto: 200
Datos extraidos con exito


13:25:31.093 | INFO    | Task run 'extract_data-c5c' - Finished in state Completed()

Datos limpiados con exito


13:25:32.350 | INFO    | Task run 'clean_data-b13' - Finished in state Completed()

Error al cargar los datos en la base de datos: Error binding parameter 1: type 'Timestamp' is not supported


13:25:32.356 | INFO    | Task run 'load_data-13c' - Finished in state Completed()

13:25:32.862 | INFO    | Flow run 'weightless-cuttlefish' - Finished in state Completed()